# Telugu LoRA vs QLoRA — Kaggle evaluation notebook

Runs `scripts/evaluate.py` (vLLM + sacrebleu) on FLORES+ devtest, eng_Latn → tel_Telu.

Kaggle settings: **Accelerator = GPU T4** · **Internet = On** · Secret `HF_TOKEN` attached.
vLLM gets its own venv (separate from Axolotl's) to avoid dependency conflicts.

## 1. Environment check

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

## 2. Install vLLM into an isolated venv

In [ ]:
%%bash
set -e
pip install -q uv
uv venv --python 3.12 /tmp/vllm-venv
uv pip install --python /tmp/vllm-venv/bin/python --torch-backend auto     "vllm==0.30.0" "sacrebleu==2.6.0" sentencepiece
/tmp/vllm-venv/bin/python -c "import torch, vllm, sacrebleu; print('torch', torch.__version__, '| vllm', vllm.__version__, '| sacrebleu', sacrebleu.__version__)"

## 3. Secrets and GPU selection

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("HF_TOKEN loaded:", bool(os.environ["HF_TOKEN"]))

## 4. Get the code (public repo)

In [ ]:
%%bash
rm -rf /kaggle/working/telugulora
git clone --depth 1 https://github.com/pashamgithub/telugulora.git /kaggle/working/telugulora
cd /kaggle/working/telugulora && git log --oneline -1 && ls scripts

## 5. Quick test — 20 sentences (catches errors in ~minutes)

In [ ]:
%%bash
cd /kaggle/working/telugulora
/tmp/vllm-venv/bin/python scripts/evaluate.py --name baseline-smoke --limit 20

## 6. Full baseline — all 1012 devtest sentences

In [ ]:
%%bash
cd /kaggle/working/telugulora
/tmp/vllm-venv/bin/python scripts/evaluate.py --name baseline

## 7. Inspect and package results

In [ ]:
import json
rows = [json.loads(l) for l in open("/kaggle/working/telugulora/results/baseline/predictions.jsonl", encoding="utf-8")]
for r in rows[:5]:
    print("EN :", r["en"]); print("REF:", r["ref"]); print("HYP:", r["hyp"]); print()

In [ ]:
%%bash
cd /kaggle/working/telugulora && zip -r /kaggle/working/results-baseline.zip results/baseline
echo "Download results-baseline.zip from the Output panel"